In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import fnmatch
from connect import bob
from limb_fitting import *
from utils import *
from fit_cld import *
from scipy.ndimage import gaussian_filter

In [2]:
sftp = bob()

top_dir = '/data/slam/valori/test_l2_fmdb/FDT_test_release_v08_2025/v4/l2/'
#top_dir = '/data/solo/phi/data/fmdb/l1/'
dirs = sorted(sftp.listdir(top_dir))

Q = []

output_file = 'cld_fit.csv'

with open(output_file, 'w') as f:
    f.write('date, did, alpha, beta, epsilon, scale, bias, drsun, sigma\n')

for directory in dirs:
    #if fnmatch.fnmatch(directory, '2024*') or fnmatch.fnmatch(directory, '2025*'):
    if fnmatch.fnmatch(directory, '2024-01*'):
        for file in sorted(sftp.listdir(top_dir + directory)):
            if fnmatch.fnmatch(file, '*stokes*.fits.gz'):
                print(file)

                remote_file = top_dir + directory + '/' + file
                local_file = 'temp.fits.gz'
                sftp.get(remote_file, local_file)

                stop

solo_L2_phi-fdt-stokes_20240101T040003_V202602220902_0441010503.fits.gz


NameError: name 'stop' is not defined

In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xd, yd = s['xd'], s['yd']

file = 'temp.fits.gz'

with fits.open(file) as hdul:
    header = hdul[0].header
    data = hdul[0].data

cpos = header['CONTPOS'] - 1
xr, yr = reflection_point_predict(header)

image = data[cpos,0].copy()
#ghost = gaussian_filter(reflect(image, xr, yr), 8)
#image = image - ghost * 0.01

image = undistort(image, header, xd, yd)
xc, yc, rsun = find_center(image)
xr, yr = reflection_point_predict(header)

phi = np.arctan((yc - yr) / (xc - xr)) * 180 / np.pi

print(xc, yc, rsun)

358.6003363394623 395.5085696898807 282.1833437991221


In [322]:
from scipy.optimize import least_squares

alpha = 1

def residuals(args, image):
    beta, epsilon, bias = args
    r, profile = scan(image, r0=rsun + 10, h=200, phi0=phi-90, phi1=phi+90, xc=xc, yc=yc, rsun=rsun)
    image_ = remove_straylight(image, alpha=alpha, beta=beta, epsilon=epsilon, niter=1) - bias
    r_, profile_ = scan(image_, r0=rsun + 10, h=200, phi0=phi-90, phi1=phi+90, xc=xc, yc=yc, rsun=rsun)
    return np.nan_to_num(profile_ / profile)


result = least_squares(residuals, np.array([1., 0.1, 0]),
                           bounds=([0.5, 0, -1e-2], [3, 1, 1e-2]),
                           args=(image,))


In [323]:
beta, epsilon, bias = result.x
print(beta, epsilon, bias)

image_ = remove_straylight(image, alpha=alpha, beta=beta, epsilon=epsilon, niter=1) - bias

0.8879492558156504 0.05939302293373576 -0.0018242458741973735


In [324]:
plt.figure(figsize=(10,10))

r, profile = scan(image, r0=rsun, h=200, phi0=phi-90, phi1=phi+90)
plt.plot(r, profile)

r_, profile_ = scan(image_, r0=rsun, h=200, phi0=phi-90, phi1=phi+90)
plt.plot(r_, profile_)


plt.xlim(rsun-20, rsun+200)
plt.ylim(-0.01,0.01)
plt.grid(True)
plt.tight_layout()

In [325]:
plt.figure(figsize=(10,10))
plt.imshow(image_, cmap='inferno', vmin=-0.01, vmax=0.01)
plt.tight_layout()

In [330]:
plt.figure(figsize=(10,10))
plt.imshow(image1 - image_, cmap='seismic', vmin=-0.05, vmax=0.05)
plt.tight_layout()

In [321]:
image1 = image_.copy()